# Discovery, Validation, And Artifact Inspection

This notebook uses the shared Drive dataset as a read-only source, uses local Colab artifacts for reliability, and exports the finished discovery/validation outputs to a clearly separate Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import os
import subprocess
import yaml
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

def run_and_stream(command, *, cwd=None):
    env = dict(os.environ)
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, inspection, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'artifacts' / 'colab_runtime_experiment.yaml'

USE_FULL_DATASET = False
TARGET_EXPERIENCE_LEVEL = 'Familiar'
MAX_SESSIONS = 10

selection_disabled = {
    'output_name': 'full_dataset_colab',
    'session_ids': [],
    'subject_ids': [],
    'exclude_session_ids': [],
    'exclude_subject_ids': [],
    'session_ids_file': None,
    'subject_ids_file': None,
    'exclude_session_ids_file': None,
    'exclude_subject_ids_file': None,
    'experience_levels': [],
    'session_types': [],
    'image_sets': [],
    'session_numbers': [],
    'project_codes': [],
    'brain_regions_any': [],
    'min_n_units': None,
    'max_n_units': None,
    'min_trial_count': None,
    'max_trial_count': None,
    'min_duration_s': None,
    'max_duration_s': None,
    'split_seed': None,
    'split_primary_axis': None,
    'train_fraction': None,
    'valid_fraction': None,
    'discovery_fraction': None,
    'test_fraction': None,
}

experiment_payload = yaml.safe_load(base_experiment_config.read_text(encoding='utf-8'))
catalog_csv = repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv'
if USE_FULL_DATASET:
    experiment_payload['dataset_selection'] = selection_disabled
else:
    catalog = pd.read_csv(catalog_csv)
    subset = catalog.loc[catalog['experience_level'] == TARGET_EXPERIENCE_LEVEL].sort_values('session_id').head(MAX_SESSIONS)
    assert len(subset) > 0, f'No sessions found for experience_level={TARGET_EXPERIENCE_LEVEL}'
    session_ids_file = repo_root / 'artifacts' / f'{TARGET_EXPERIENCE_LEVEL.lower()}_{MAX_SESSIONS}_session_ids.txt'
    session_ids_file.write_text('\n'.join(subset['session_id'].astype(str).tolist()) + '\n', encoding='utf-8')
    experiment_payload['dataset_selection'] = {
        'output_name': f'{TARGET_EXPERIENCE_LEVEL.lower()}_{MAX_SESSIONS}_session_subset',
        'session_ids': [],
        'subject_ids': [],
        'exclude_session_ids': [],
        'exclude_subject_ids': [],
        'session_ids_file': str(session_ids_file),
        'subject_ids_file': None,
        'exclude_session_ids_file': None,
        'exclude_subject_ids_file': None,
        'experience_levels': [],
        'session_types': [],
        'image_sets': [],
        'session_numbers': [],
        'project_codes': [],
        'brain_regions_any': [],
        'min_n_units': None,
        'max_n_units': None,
        'min_trial_count': None,
        'max_trial_count': None,
        'min_duration_s': None,
        'max_duration_s': None,
        'split_seed': 7,
        'split_primary_axis': 'session',
        'train_fraction': 0.6,
        'valid_fraction': 0.2,
        'discovery_fraction': 0.1,
        'test_fraction': 0.1,
    }

experiment_payload.setdefault('training', {})['log_every_steps'] = 1

runtime_experiment_config.write_text(yaml.safe_dump(experiment_payload, sort_keys=False), encoding='utf-8')
experiment_config = runtime_experiment_config

dataset_selection_active = any(
    value not in (None, [], '', {})
    for key, value in experiment_payload.get('dataset_selection', {}).items()
    if key != 'output_name'
)

checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
selected_checkpoint = checkpoint_dir / 'pcc_best.pt'
if not selected_checkpoint.exists():
    epoch_candidates = sorted(checkpoint_dir.glob('pcc_epoch_*.pt'))
    assert epoch_candidates, 'No checkpoint found in repo/artifacts/checkpoints. Run the training notebook first.'
    selected_checkpoint = epoch_candidates[-1]
run_label = datetime.now().strftime('discover_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)
if not USE_FULL_DATASET:
    print('Target experience level:', TARGET_EXPERIENCE_LEVEL)
    print('Max sessions:', MAX_SESSIONS)
print('Selected checkpoint:', selected_checkpoint)
print('Drive export path:', drive_run_export_root)

In [ ]:
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    run_and_stream([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], cwd=repo_root)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

In [ ]:
reporter.begin('discovery', next_artifact='local discovery artifact + cluster summary')
%cd {repo_root}
run_and_stream([
    'pcc-discover',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--split', 'discovery',
], cwd=repo_root)
reporter.finish('discovery')

In [ ]:
discovery_artifact = Path(selected_checkpoint).with_name(f"{Path(selected_checkpoint).stem}_discovery_discovery.json")
cluster_summary_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.json")
cluster_summary_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.csv")

reporter.begin('validation', next_artifact='local validation summary json/csv')
%cd {repo_root}
run_and_stream([
    'pcc-validate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--discovery-artifact', str(discovery_artifact),
], cwd=repo_root)
reporter.finish('validation')

In [ ]:
validation_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.json")
validation_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.csv")

with open(discovery_artifact, 'r', encoding='utf-8') as handle:
    discovery_payload = json.load(handle)
with open(cluster_summary_json, 'r', encoding='utf-8') as handle:
    cluster_summary_payload = json.load(handle)
with open(validation_json, 'r', encoding='utf-8') as handle:
    validation_payload = json.load(handle)

display(pd.read_csv(cluster_summary_csv))
display(pd.DataFrame(discovery_payload['candidates']).head())
display(pd.read_csv(validation_csv))
print('Cluster count:', cluster_summary_payload['cluster_count'])
print('Recurrence summary:', validation_payload['recurrence_summary'])

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')